In [1]:
# Cell 1 — Mount Drive and install packages
from google.colab import drive
drive.mount('/content/drive')
DRIVE_BASE = '/content/drive/MyDrive/PhishGuard'

import os, subprocess
subprocess.run(['pip', 'install',
    'xgboost==2.1.0', 'scikit-learn==1.5.0',
    'pandas==2.2.0', 'numpy==1.26.4', 'joblib==1.4.0', '-q'], check=False)

%pip install pyevmasm==0.2.3 -q

assert os.path.exists(f'{DRIVE_BASE}/features/contract_features.csv'), \
    'contract_features.csv not found — run Notebook 03 first'
assert os.path.exists(f'{DRIVE_BASE}/models/contract_feature_schema.json'), \
    'contract_feature_schema.json not found — run Notebook 03 first'
print('Cell 1 ready.')


In [2]:
# Cell 2 — Load features and schema, validate inputs
import pandas as pd
import numpy as np
import json
import joblib
import os
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score, confusion_matrix,
                             classification_report)
import xgboost as xgb
from xgboost import XGBClassifier

MODEL_PATH  = f'{DRIVE_BASE}/models/contract_xgboost_v1.pkl'
SCHEMA_PATH = f'{DRIVE_BASE}/models/contract_feature_schema.json'
EVAL_DIR    = f'{DRIVE_BASE}/evaluation'

with open(SCHEMA_PATH) as f:
    FEATURE_COLS = json.load(f)

df = pd.read_csv(f'{DRIVE_BASE}/features/contract_features.csv')

TOTAL  = len(df)
PHISH  = (df['label'] == 1).sum()
BENIGN = (df['label'] == 0).sum()

assert df.shape[1] == 23,              f'Wrong column count: {df.shape[1]}'
assert PHISH == BENIGN,                f'Imbalanced: phishing={PHISH} benign={BENIGN}'
assert TOTAL >= 1000,                  f'Too few rows: {TOTAL}'
assert df.isnull().sum().sum() == 0,   'Nulls found'
assert len(FEATURE_COLS) == 21,        f'Schema length wrong: {len(FEATURE_COLS)}'
assert all(c in df.columns for c in FEATURE_COLS), 'Missing feature columns'

X = df[FEATURE_COLS].values
y = df['label'].values

assert X.shape[1] == len(FEATURE_COLS), \
    f'Feature count mismatch: {X.shape[1]} vs {len(FEATURE_COLS)}'

print(f'Loaded: {df.shape}')
print(f'Features: {len(FEATURE_COLS)}')
print(f'Rows: {TOTAL} | Phishing: {PHISH} | Benign: {BENIGN}')
print(f'Training features:')
for f in FEATURE_COLS:
    print(f'  {f}')
print('Cell 2 validation passed.')


In [22]:
# Cell 3 — Train/test split
all_idx = np.arange(len(y))
train_idx, test_idx = train_test_split(
    all_idx, test_size=0.30, stratify=y, random_state=42)

X_train, X_test = X[train_idx], X[test_idx]
y_train, y_test = y[train_idx], y[test_idx]

os.makedirs(f'{DRIVE_BASE}/models', exist_ok=True)
np.save(f'{DRIVE_BASE}/models/contract_test_indices.npy', test_idx)
print(f'Saved contract_test_indices.npy — {len(test_idx)} rows')

print(f'Train: {X_train.shape} | phishing: {y_train.sum()} | benign: {(y_train==0).sum()}')
print(f'Test:  {X_test.shape}  | phishing: {y_test.sum()}  | benign: {(y_test==0).sum()}')
assert len(test_idx)  == 390, f'Test size wrong: {len(test_idx)}'
assert len(train_idx) == 910, f'Train size wrong: {len(train_idx)}'


In [4]:
# Cell 4 — XGBoost training with 5-fold cross-validation
scale_pos_weight = float((y_train == 0).sum()) / float((y_train == 1).sum())
print(f'scale_pos_weight: {scale_pos_weight:.4f}')

model = XGBClassifier(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.05,
    scale_pos_weight=scale_pos_weight,
    eval_metric='aucpr',
    tree_method='hist',
    device='cpu',
    random_state=42
)

cv     = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scores = cross_val_score(model, X_train, y_train,
                         cv=cv, scoring='average_precision', n_jobs=1)
print(f'CV PR-AUC per fold: {np.round(scores, 4)}')
print(f'Mean CV PR-AUC: {scores.mean():.4f} ± {scores.std():.4f}')

if scores.mean() < 0.75:
    print('WARNING: Below 0.75 target. Debug before proceeding.')
else:
    print('CV target met. Proceed to Cell 5.')


In [5]:
# Fit final model on full training set
model.fit(X_train, y_train)
print('Model fitted on training set.')

In [7]:
# Cell 5 — Evaluation on held-out test set
y_pred      = model.predict(X_test)
y_pred_prob = model.predict_proba(X_test)[:, 1]

acc  = accuracy_score(y_test, y_pred)
prec = precision_score(y_test, y_pred)
rec  = recall_score(y_test, y_pred)
f1   = f1_score(y_test, y_pred)
auc  = roc_auc_score(y_test, y_pred_prob)
cm   = confusion_matrix(y_test, y_pred)

print('=== TEST SET EVALUATION ===')
print(f'  Accuracy:  {acc:.4f}')
print(f'  Precision: {prec:.4f}')
print(f'  Recall:    {rec:.4f}')
print(f'  F1:        {f1:.4f}')
print(f'  ROC-AUC:   {auc:.4f}')
print(f'\nConfusion Matrix:\n{cm}')
print(f'\n{classification_report(y_test, y_pred, target_names=["benign","phishing"])}')

assert f1  >= 0.70, f'F1 too low: {f1:.4f}'
assert auc >= 0.75, f'AUC too low: {auc:.4f}'
print('Minimum performance thresholds passed.')



In [10]:
# Cell 7 — Save model, optimal threshold, and verify
from sklearn.metrics import precision_recall_curve
import json

# ── Find optimal threshold automatically ─────────────────────
precision_vals, recall_vals, thresholds = precision_recall_curve(
    y_test, y_pred_prob)

best_f1        = 0
best_threshold = 0.5

for p, r, t in zip(precision_vals, recall_vals, thresholds):
    f1 = 2 * p * r / (p + r) if (p + r) > 0 else 0
    if p >= 0.80 and r >= 0.80 and f1 > best_f1:
        best_f1        = f1
        best_threshold = float(t)

print(f'Optimal threshold: {best_threshold:.4f}')
print(f'Best F1 at threshold: {best_f1:.4f}')

# ── Apply optimal threshold and show final results ────────────
y_pred_tuned = (y_pred_prob >= best_threshold).astype(int)
print(f'\n=== FINAL CLASSIFICATION REPORT (threshold={best_threshold:.4f}) ===')
print(classification_report(y_test, y_pred_tuned,
      target_names=['benign', 'phishing']))

# ── Save model ────────────────────────────────────────────────
MODEL_PATH = f'{DRIVE_BASE}/models/contract_xgboost_v1.pkl'
joblib.dump(model, MODEL_PATH)
print(f'Saved: {MODEL_PATH}')

# ── Save threshold ────────────────────────────────────────────
THRESHOLD_PATH = f'{DRIVE_BASE}/models/contract_threshold.json'
with open(THRESHOLD_PATH, 'w') as f:
    json.dump({
        'threshold': best_threshold,
        'best_f1':   best_f1,
        'note':      'Optimal threshold from PR curve. Use instead of default 0.5 for production predictions.'
    }, f, indent=2)
print(f'Saved: {THRESHOLD_PATH}')

# ── Verify round-trip ─────────────────────────────────────────
loaded     = joblib.load(MODEL_PATH)
test_preds = loaded.predict_proba(X_test[:5])[:, 1]
saved_t    = json.load(open(THRESHOLD_PATH))['threshold']
print(f'Verify predictions: {np.round(test_preds, 4)}')
print(f'Verify threshold:   {saved_t}')
print('Model and threshold saved and verified.')
print('Notebook 05 complete.')